# STAC Raster Fetch and Occupied-H3 Clip

Uses the consolidated Puerto Rico STAC catalog from `05_pr_raster_catalog_indexes.py`
to fetch every intersecting raster item for occupied H3 cells in San Juan and
Isabela, clips each asset to the occupied cell geometry, reprojects to
EPSG:3857, and writes model-ready local derivatives under
`data/rasters/stac/local/`.

In [ ]:
"""07_pr_stac_municipality_fetch.py

Fetch and clip intersecting STAC raster assets for occupied H3 cells.
"""

from __future__ import annotations

import json
import os
import sys
from pathlib import Path

from time import sleep

import duckdb
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from dotenv import load_dotenv
from rasterio.mask import mask
from rasterio.transform import array_bounds
from rasterio.warp import calculate_default_transform, reproject, transform_geom
from shapely.geometry import mapping
from tqdm import tqdm


def resolve_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if any((candidate / marker).exists() for marker in ("project_rules.md", ".git")):
            return candidate
    return current


PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")
from utils.overture import occupied_h3_cells_sql

TARGET_MUNICIPALITIES = ("San Juan", "Isabela")
OVERTURE_BUILDINGS_TABLE = "pr_overture_buildings"
STAC_CATALOG_PATH = PROJECT_ROOT / "data" / "rasters" / "stac" / "pr_raster_catalog_items.parquet"
LOCAL_STAC_ROOT = PROJECT_ROOT / "data" / "rasters" / "stac" / "local"
LOCAL_FETCH_MANIFEST_PATH = PROJECT_ROOT / "data" / "rasters" / "stac" / "pr_local_stac_fetch_manifest.parquet"
MODEL_READY_CRS = "EPSG:3857"
PREFERRED_ASSET_COLUMNS = (
    ("visual_asset_href", "visual"),
    ("analytic_asset_href", "analytic"),
)
MAX_ITEMS_THIS_RUN = int(os.getenv("MAX_STAC_ITEMS_THIS_RUN", "300")) or None
OVERWRITE_EXISTING = os.getenv("OVERWRITE_STAC_LOCAL_CACHE", "0") == "1"


def resolve_db_path() -> Path:
    value = os.getenv("VECTOR_DB")
    if value:
        path = Path(value)
        if not path.is_absolute():
            path = PROJECT_ROOT / path if len(path.parts) > 1 else PROJECT_ROOT / "data" / "vectors" / path
        return path
    return PROJECT_ROOT / "data" / "PR_PV_plan_data.duckdb"


def _to_bytes(value: object) -> bytes:
    if isinstance(value, memoryview):
        return value.tobytes()
    if isinstance(value, bytearray):
        return bytes(value)
    if isinstance(value, bytes):
        return value
    return bytes(value)


def slugify(value: str) -> str:
    return value.strip().replace(" ", "_").replace("/", "_")


def table_exists(con: duckdb.DuckDBPyConnection, table_name: str) -> bool:
    row = con.execute(
        "SELECT COUNT(*) FROM information_schema.tables WHERE table_name = ?;",
        [table_name],
    ).fetchone()
    return bool(row and row[0])


def load_target_h3_cells(con: duckdb.DuckDBPyConnection) -> gpd.GeoDataFrame:
    if not table_exists(con, OVERTURE_BUILDINGS_TABLE):
        raise RuntimeError(
            f"{OVERTURE_BUILDINGS_TABLE} not found; run notebooks/vectors/03_overture_buildings_ingest.py first."
        )

    municipio_sql = ", ".join(f"'{municipio}'" for municipio in TARGET_MUNICIPALITIES)
    occupied_h3_sql = occupied_h3_cells_sql(OVERTURE_BUILDINGS_TABLE)
    frame = con.execute(
        f"""
        WITH occupied_h3 AS ({occupied_h3_sql})
        SELECT
               h3_cell_id,
               h3_resolution,
               municipality_name AS municipio,
               municipality_geoid AS municipio_geoid,
               building_count,
               municipality_building_count,
               crosses_municipality_boundary,
               ST_AsWKB(geometry) AS geometry_wkb
         FROM occupied_h3
        WHERE municipality_name IN ({municipio_sql})
        ORDER BY municipio, h3_cell_id;
        """
    ).fetchdf()
    if frame.empty:
        return gpd.GeoDataFrame(
            columns=[
                "h3_cell_id",
                "h3_resolution",
                "municipio",
                "municipio_geoid",
                "building_count",
                "municipality_building_count",
                "crosses_municipality_boundary",
                "geometry",
            ],
            geometry="geometry",
            crs="EPSG:4326",
        )

    geometry = gpd.GeoSeries.from_wkb(frame["geometry_wkb"].map(_to_bytes), crs="EPSG:4326")
    return gpd.GeoDataFrame(frame.drop(columns=["geometry_wkb"]), geometry=geometry, crs="EPSG:4326")


def load_catalog() -> gpd.GeoDataFrame:
    if not STAC_CATALOG_PATH.exists():
        raise RuntimeError(
            f"STAC catalog not found at {STAC_CATALOG_PATH}; run notebooks/rasters/05_pr_raster_catalog_indexes.py first."
        )
    catalog = gpd.read_parquet(STAC_CATALOG_PATH)
    if catalog.crs is None:
        return catalog.set_crs("EPSG:4326")
    return catalog.to_crs("EPSG:4326")


def choose_asset(row: pd.Series) -> tuple[str | None, str | None]:
    for column_name, asset_role in PREFERRED_ASSET_COLUMNS:
        href = row.get(column_name)
        if isinstance(href, str) and href.strip():
            return href, asset_role
    return None, None


def enumerate_target_items(
    catalog: gpd.GeoDataFrame,
    occupied_h3_cells: gpd.GeoDataFrame,
) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    if catalog.empty or occupied_h3_cells.empty:
        return pd.DataFrame()

    ranked = catalog.copy()
    ranked["acquired_at"] = pd.to_datetime(ranked["acquired_at"], utc=True, errors="coerce")
    ranked = ranked.sort_values(
        by=["source", "acquired_at", "gsd", "item_id"],
        ascending=[True, False, True, True],
        na_position="last",
    ).reset_index(drop=True)

    candidate_rows: list[dict[str, object]] = []
    for _, item in ranked.iterrows():
        asset_href, asset_role = choose_asset(item)
        if asset_href is None:
            continue
        candidate_rows.append(
            {
                "source": item["source"],
                "item_id": item["item_id"],
                "asset_role": asset_role,
                "asset_href": asset_href,
                "acquired_at": item.get("acquired_at"),
                "gsd": item.get("gsd"),
                "platform": item.get("platform"),
                "catalog_self_href": item.get("self_href"),
                "geometry": item.geometry,
            }
        )

    if not candidate_rows:
        return pd.DataFrame()

    asset_candidates = gpd.GeoDataFrame(candidate_rows, geometry="geometry", crs="EPSG:4326")
    joined = gpd.sjoin(
        asset_candidates,
        occupied_h3_cells,
        how="inner",
        predicate="intersects",
        lsuffix="asset",
        rsuffix="cell",
    )
    if joined.empty:
        return pd.DataFrame()

    # set occupied_h3_cells index to h3_cell_id for easy lookup of geometry during clipping;
    occupied_h3_cells = occupied_h3_cells.set_index("h3_cell_id", drop=False)
    clip_geometries = occupied_h3_cells.geometry.copy()
    # print(f"Spatially Joined gdf has columns: {joined.columns.tolist()} and index: {joined.index}")
    # print(f"Clip geometries Geoseries can be indexed by h3_cell_id: {clip_geometries.index}") 
    for idx, row in joined.iterrows():
        local_dir = (
            LOCAL_STAC_ROOT
            / slugify(str(row["source"]))
            / slugify(str(row["municipio"]))
            / str(row["h3_cell_id"])
        )
        local_path = local_dir / f"{row['item_id']}_{row['asset_role']}_epsg3857.tif"
        rows.append(
            {
                "source": row["source"],
                "item_id": row["item_id"],
                "municipio": row["municipio"],
                "municipio_geoid": row["municipio_geoid"],
                "h3_cell_id": row["h3_cell_id"],
                "h3_resolution": row["h3_resolution"],
                "building_count": row["building_count"],
                "municipality_building_count": row["municipality_building_count"],
                "crosses_municipality_boundary": row["crosses_municipality_boundary"],
                "asset_role": row["asset_role"],
                "asset_href": row["asset_href"],
                "local_asset_path": str(local_path),
                "acquired_at": row.get("acquired_at"),
                "gsd": row.get("gsd"),
                "platform": row.get("platform"),
                "catalog_self_href": row.get("catalog_self_href"),
                # the geometry of the h3 cell that will be used for clipping; 
                "geometry": clip_geometries.loc[row["h3_cell_id"]],
            }
        )

    targets = pd.DataFrame(rows)
    if targets.empty:
        return targets
    targets = targets.sort_values(
        by=["municipio", "h3_cell_id", "source", "acquired_at", "item_id", "asset_role"],
        ascending=[True, True, True, False, True, True],
        na_position="last",
    ).reset_index(drop=True)
    if MAX_ITEMS_THIS_RUN is not None:
        targets = targets.head(MAX_ITEMS_THIS_RUN).reset_index(drop=True)
    return targets


def reproject_to_model_ready_crs(
    clipped_image: np.ndarray,
    clipped_transform: rasterio.Affine,
    *,
    src_crs: rasterio.crs.CRS,
    dst_crs: str = MODEL_READY_CRS,
    nodata: float | int | None,
) -> tuple[np.ndarray, rasterio.Affine, str, float | int | None]:
    if str(src_crs).upper() == dst_crs.upper():
        return clipped_image, clipped_transform, dst_crs, nodata

    if nodata is None:
        nodata = 0 if np.issubdtype(clipped_image.dtype, np.integer) else np.nan

    bounds = array_bounds(clipped_image.shape[1], clipped_image.shape[2], clipped_transform)
    dst_transform, dst_width, dst_height = calculate_default_transform(
        src_crs,
        dst_crs,
        clipped_image.shape[2],
        clipped_image.shape[1],
        *bounds,
    )
    if dst_width <= 0 or dst_height <= 0:
        raise RuntimeError("Reprojected raster window is empty.")

    destination = np.full((clipped_image.shape[0], dst_height, dst_width), nodata, dtype=clipped_image.dtype)
    for band_idx in range(clipped_image.shape[0]):
        reproject(
            source=clipped_image[band_idx],
            destination=destination[band_idx],
            src_transform=clipped_transform,
            src_crs=src_crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            src_nodata=nodata,
            dst_nodata=nodata,
            resampling=Resampling.bilinear,
        )
    return destination, dst_transform, dst_crs, nodata


def clip_item_to_h3_cell(row: pd.Series) -> dict[str, object]:
    local_path = Path(str(row["local_asset_path"]))
    local_path.parent.mkdir(parents=True, exist_ok=True)
    sidecar_path = local_path.with_name(f"{local_path.stem}_meta.json")
    status = "reused"
    error: str | None = None
    native_proj_epsg: int | None = None
    model_ready_proj_epsg: int | None = None

    if OVERWRITE_EXISTING or not local_path.exists():
        try:
            with rasterio.Env(AWS_NO_SIGN_REQUEST="YES"):
                with rasterio.open(str(row["asset_href"])) as src:
                    if src.crs is None:
                        raise RuntimeError("Source raster is missing CRS metadata.")
                    geometry = mapping(row["geometry"])
                    if src.crs and str(src.crs).upper() != "EPSG:4326":
                        geometry = transform_geom("EPSG:4326", src.crs, geometry, precision=6)
                    clipped_image, clipped_transform = mask(src, [geometry], crop=True)
                    if clipped_image.shape[1] == 0 or clipped_image.shape[2] == 0:
                        raise RuntimeError("Clip produced an empty raster window.")

                    native_proj_epsg = src.crs.to_epsg() if src.crs else None
                    clipped_image, clipped_transform, _, nodata = reproject_to_model_ready_crs(
                        clipped_image,
                        clipped_transform,
                        src_crs=src.crs,
                        dst_crs=MODEL_READY_CRS,
                        nodata=src.nodata,
                    )
                    model_ready_proj_epsg = rasterio.crs.CRS.from_string(MODEL_READY_CRS).to_epsg()
                    meta = src.meta.copy()
                    meta.update(
                        driver="GTiff",
                        height=clipped_image.shape[1],
                        width=clipped_image.shape[2],
                        transform=clipped_transform,
                        crs=MODEL_READY_CRS,
                        compress="deflate",
                        tiled=True,
                    )
                    if nodata is not None and not (isinstance(nodata, float) and np.isnan(nodata)):
                        meta["nodata"] = nodata
                    else:
                        meta.pop("nodata", None)
                    with rasterio.open(local_path, "w", **meta) as dst:
                        dst.write(clipped_image)
            status = "fetched"
        except Exception as exc:
            status = "error"
            error = str(exc)

    if status != "error":
        acquired_at = row.get("acquired_at")
        if pd.notna(acquired_at):
            acquired_at = pd.Timestamp(acquired_at).isoformat()
        else:
            acquired_at = None
        sidecar_path.write_text(
            json.dumps(
                {
                    "source": row["source"],
                    "item_id": row["item_id"],
                    "municipio": row["municipio"],
                    "municipio_geoid": row["municipio_geoid"],
                    "h3_cell_id": row["h3_cell_id"],
                    "h3_resolution": int(row["h3_resolution"]),
                    "building_count": int(row["building_count"]),
                    "municipality_building_count": int(row["municipality_building_count"]),
                    "crosses_municipality_boundary": bool(row["crosses_municipality_boundary"]),
                    "asset_role": row["asset_role"],
                    "acquired_at": acquired_at,
                    "gsd": row.get("gsd"),
                    "platform": row.get("platform"),
                    "original_asset_href": row["asset_href"],
                    "catalog_self_href": row.get("catalog_self_href"),
                    "clip_strategy": "occupied_h3_cell",
                    "native_proj_epsg": native_proj_epsg,
                    "model_ready_proj_epsg": model_ready_proj_epsg or 3857,
                    "local_asset_path": str(local_path.relative_to(PROJECT_ROOT)),
                },
                indent=2,
            )
        )

    return {
        "source": row["source"],
        "item_id": row["item_id"],
        "municipio": row["municipio"],
        "municipio_geoid": row["municipio_geoid"],
        "h3_cell_id": row["h3_cell_id"],
        "h3_resolution": row["h3_resolution"],
        "building_count": row["building_count"],
        "municipality_building_count": row["municipality_building_count"],
        "crosses_municipality_boundary": row["crosses_municipality_boundary"],
        "asset_role": row["asset_role"],
        "asset_href": row["asset_href"],
        "local_asset_path": str(local_path.relative_to(PROJECT_ROOT)),
        "status": status,
        "error": error,
        "acquired_at": row.get("acquired_at"),
        "gsd": row.get("gsd"),
        "platform": row.get("platform"),
        "model_ready_proj_epsg": model_ready_proj_epsg or 3857,
    }

In [2]:
if __name__ == "__main__":
    con = duckdb.connect(str(resolve_db_path()))
    con.execute("INSTALL spatial; LOAD spatial;")
    con.execute("LOAD h3;")

    occupied_h3_cells = load_target_h3_cells(con)
    print(f"loaded {len(occupied_h3_cells):,} occupied H3 cells for target municipalities")
    if occupied_h3_cells.empty:
        print("no occupied H3 cells found in DuckDB.")
        con.close()
        sys.exit(0)

    catalog = load_catalog()
    print(f"loaded {len(catalog):,} STAC catalog items from {STAC_CATALOG_PATH}")

    targets = enumerate_target_items(catalog, occupied_h3_cells)
    print(f"intersecting occupied-H3 clips queued: {len(targets):,}")
    if targets.empty:
        print("no intersecting STAC assets were found for the occupied H3 cells.")
        con.close()
        sys.exit(0)

    print(targets.groupby(["municipio", "source", "asset_role"]).size().to_string())

    manifest_rows = [clip_item_to_h3_cell(row) for _, row in tqdm(targets.iterrows(), total=len(targets), desc="Processing STAC items", unit="clipped assets")]
    manifest = pd.DataFrame(manifest_rows)
    manifest_rows = []
    for _, row in tqdm(targets.iterrows(), total=len(targets), desc="Processing STAC items", unit="clipped assets"):
        manifest_rows.append(clip_item_to_h3_cell(row))
        sleep(0.25)
    manifest = pd.DataFrame(manifest_rows)
    LOCAL_FETCH_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    manifest.to_parquet(LOCAL_FETCH_MANIFEST_PATH, index=False)
    print(f"wrote {len(manifest):,} fetch rows to {LOCAL_FETCH_MANIFEST_PATH}")
    print(manifest.groupby(["municipio", "source", "status"]).size().to_string())
    con.close()

loaded 61,835 occupied H3 cells for target municipalities
loaded 649 STAC catalog items from /home/asvnpr/Documents/repos/PLAN6068_PV_Project/data/rasters/stac/pr_raster_catalog_items.parquet
intersecting occupied-H3 clips queued: 150
municipio  source   asset_role
Isabela    pr_naip  visual        150


Processing STAC items: 100%|██████████| 150/150 [00:41<00:00,  3.63clipped assets/s]

wrote 150 fetch rows to /home/asvnpr/Documents/repos/PLAN6068_PV_Project/data/rasters/stac/pr_local_stac_fetch_manifest.parquet
municipio  source   status
Isabela    pr_naip  error       3
                    reused    147
